<a href="https://colab.research.google.com/github/nishant-tyagi-ainishant-tyagi-ai/chromadb-rag-pipeline/blob/main/hyde_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers groq langchain-groq

In [ ]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("Key set!")

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM setup
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup done!")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

docs = [
    Document(page_content="RAG stands for Retrieval Augmented Generation. It combines a retriever and a generator to answer questions using external knowledge."),
    Document(page_content="ChromaDB is a vector database used to store and retrieve embeddings efficiently for semantic search."),
    Document(page_content="HyDE stands for Hypothetical Document Embeddings. It generates a fake answer to a query and uses its embedding for retrieval."),
    Document(page_content="LangChain is a framework for building LLM-powered applications including RAG pipelines, agents, and chatbots."),
    Document(page_content="Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings."),
    Document(page_content="Query expansion improves retrieval by generating multiple variations of the original query to capture different perspectives."),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="hyde_demo")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vectorstore ready!")

In [ ]:
hyde_prompt = ChatPromptTemplate.from_template("""
You are an expert. Given the question below, write a short hypothetical passage
that would directly answer this question. Write only the passage, nothing else.

Question: {question}
Hypothetical Answer:""")

hyde_chain = hyde_prompt | llm | StrOutputParser()

answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def hyde_retrieve(question):
    hypothetical_doc = hyde_chain.invoke({"question": question})
    print(f"Hypothetical Doc:\n{hypothetical_doc}\n")
    retrieved_docs = retriever.invoke(hypothetical_doc)
    return retrieved_docs

rag_chain = (
    {"context": lambda x: format_docs(hyde_retrieve(x)), "question": RunnablePassthrough()}
    | answer_prompt
    | llm
    | StrOutputParser()
)

print("HyDE chain ready!")

In [ ]:
result = rag_chain.invoke("What is HyDE and how does it improve retrieval?")
print(result)

In [ ]:
result2 = rag_chain.invoke("What is ChromaDB used for?")
print(result2)